In [15]:
import os
import sys
import json
import glob
import gc

import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration

from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType
import warnings 
warnings.filterwarnings('ignore')

In [16]:
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

In [3]:
#Table with predictions
sf_pred_table = session.table('MOP_DATABASE.SOQ.TFT_F100_PREDICTIONS')

In [4]:
sf_pred_table.show()

------------------------------------------------------------------------------------------------------------------------------------------------------
|"RUN_DATE"  |"ITERATION_DETAILS"                        |"PARENT_DEALER_CODE_MODEL_FAMILY"      |"CAL_DATE"  |"PREDICTION_QUANTILE"  |"PREDICTION"  |
------------------------------------------------------------------------------------------------------------------------------------------------------
|2026-09-18  |Iteration 2 : Without calendar attributes  |11875_XTREME 125_DISC_SELF_CAST_BLACK  |2026-10-13  |70                     |1             |
|2026-09-18  |Iteration 2 : Without calendar attributes  |11875_XTREME 125_DISC_SELF_CAST_BLACK  |2026-10-14  |70                     |1             |
|2026-09-18  |Iteration 2 : Without calendar attributes  |11875_XTREME 125_DISC_SELF_CAST_BLACK  |2026-10-15  |70                     |1             |
|2026-09-18  |Iteration 2 : Without calendar attributes  |11875_XTREME 125_DISC_SELF_CAST_BLAC

In [6]:
sum_of_predictions=sf_pred_table.group_by("PREDICTION_QUANTILE").agg(F.sum("PREDICTION").alias("TOTAL_PREDICTIONS"))
sum_of_predictions.show()

-----------------------------------------------
|"PREDICTION_QUANTILE"  |"TOTAL_PREDICTIONS"  |
-----------------------------------------------
|Mean                   |1970994              |
|70                     |2419613              |
-----------------------------------------------



In [10]:
#Iteration 3 calculations
pred_itr_3 = pd.read_parquet(r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#3_feature_engineering\Modelling\predictions_2026_negbin_torch_convention\20260922_153432_34f1cdf5\DA7CA8~1.PAR")
pred_itr_3.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,CAL_DATE,PRED_MEAN,PRED_Q45,PRED_Q55,PRED_Q70,PRED_Q75
0,10915_DESTINI_DISC_SELF_CAST_BLACK,2026-09-01,0.001740,0.0,0.0,0.0,0.0
1,10915_DESTINI_DISC_SELF_CAST_BLACK,2026-09-02,0.001030,0.0,0.0,0.0,0.0
2,10915_DESTINI_DISC_SELF_CAST_BLACK,2026-09-03,0.001248,0.0,0.0,0.0,0.0
3,10915_DESTINI_DISC_SELF_CAST_BLACK,2026-09-04,0.048847,0.0,0.0,0.0,0.0
4,10915_DESTINI_DISC_SELF_CAST_BLACK,2026-09-05,0.006637,0.0,0.0,0.0,0.0


In [12]:
pred_itr_3.loc[pred_itr_3["CAL_DATE"]<='2026-09-23',:]['PRED_MEAN'].sum()

285553.42248591717

In [12]:
list_of_columns = pred_itr_3.columns.tolist()

pred_itr_3[list_of_columns[2:]].sum()


PRED_MEAN    2.073623e+06
PRED_Q45     1.167597e+06
PRED_Q55     1.523295e+06
PRED_Q70     2.313679e+06
PRED_Q75     2.703753e+06
dtype: float64

In [5]:
import datetime 
from datetime import datetime as dt
date_new=datetime.date(2026,9,22)
date_new

datetime.date(2026, 9, 22)

In [ ]:

q70_pred = pred_itr_3[["PARENT_DEALER_CODE_MODEL_FAMILY","CAL_DATE","PRED_Q70"]]
q70_pred["PREDICTION_QUANTILE"] = 70 
q70_pred["RUN_DATE"] = date_new 
q70_pred["ITERATION_DETAILS"] = "Iteration 3 : With feature engineered: without penalty cols"

order_of_columns_required = ["RUN_DATE","ITERATION_DETAILS","PARENT_DEALER_CODE_MODEL_FAMILY","CAL_DATE","PREDICTION_QUANTILE","PRED_Q70"]

q70_pred = q70_pred[order_of_columns_required]
q70_pred.rename(columns={"PRED_Q70":'PREDICTION'},inplace=True)
q70_pred_sf = session.create_dataframe(q70_pred)
q70_pred_sf.write.mode('append').save_as_table('MOP_DATABASE.SOQ.TFT_F100_PREDICTIONS')

In [8]:
mean_pred = pred_itr_3[["PARENT_DEALER_CODE_MODEL_FAMILY","CAL_DATE","PRED_MEAN"]]
mean_pred["PREDICTION_QUANTILE"] = "Mean"
mean_pred["RUN_DATE"] = date_new 
mean_pred["ITERATION_DETAILS"] = "Iteration 3 : With feature engineered: without penalty cols"

order_of_columns_required = ["RUN_DATE","ITERATION_DETAILS","PARENT_DEALER_CODE_MODEL_FAMILY","CAL_DATE","PREDICTION_QUANTILE","PRED_MEAN"]

mean_pred = mean_pred[order_of_columns_required]
mean_pred.rename(columns={"PRED_MEAN":'PREDICTION'},inplace=True)
mean_pred_sf = session.create_dataframe(mean_pred)
# mean_pred_sf.write.mode('append').save_as_table('MOP_DATABASE.SOQ.TFT_F100_PREDICTIONS')

In [9]:
mean_pred_sf.filter(F.col("CAL_DATE").between('2026-09-01','2026-09-22')).select(F.sum('PREDICTION')).show()

-------------------------
|"SUM(""PREDICTION"")"  |
-------------------------
|273177.16240965825     |
-------------------------



In [24]:
snowflake_utils.shape_of_snowpark_df(mean_pred_sf)

(3221193, 6)

In [27]:
mean_pred_sf.write.mode('overwrite').save_as_table('MOP_DATABASE.SOQ.PREDICTION_TEMP')

### Actual Sales Calculation

In [13]:
def get_ecr_sales_snowpark(session, customer_types, start_date, name_of_models,end_date):
    ecr_sales = session.table("ANALYTICS_DATABASE.ANALYTICS_SALES.CUSTOMER_RETAILS") \
        .filter(F.col("X_CUSTOMER_TYPE").in_(customer_types)) \
        .filter((F.col("CAL_DATE") >= F.lit(start_date)) & (F.col("CAL_DATE") <= F.lit(end_date)))
        
    if name_of_models is not None:
        ecr_sales = ecr_sales.filter(F.col("MODEL").isin(name_of_models))
        
    ecr_sales = ecr_sales.with_column("NET_SALES", 
    F.when(
        (F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")) < 0, 
        F.lit(0)
    ).otherwise(
        F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")
    )
)


    return ecr_sales

In [17]:
def return_models_for_forecasting(session, use_selected_models, logger):
    """
    Returns a list of model names or None based on user filter parameter settings.
    """
    if not use_selected_models:
        # logger.info("Model filter disabled — considering ALL models in ECR sales.")
        return None

    models_for_forecasting = session.table('MOP_DATABASE.SOQ.MODELS_FOR_FORECASTING').to_pandas()
    name_of_models = models_for_forecasting["MODEL_NAME"].tolist()
    # logger.info("Model filter enabled — %s models selected for forecasting.", len(name_of_models))
    return name_of_models

name_of_models = return_models_for_forecasting(session,True,None)

In [18]:
actual_ecr_sales = get_ecr_sales_snowpark(session,start_date='2026-09-01',end_date='2026-09-23',customer_types=['Individual'],name_of_models=name_of_models)

In [20]:
actual_ecr_sales = get_ecr_sales_snowpark(session,start_date='2026-09-01',end_date='2026-09-22',customer_types=['Individual'],name_of_models=name_of_models)

actual_ecr_sales = actual_ecr_sales.select('DEALER_CODE','CAL_DATE','SKU','NET_SALES')

obd_data = session.table("MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW") 
sku_supercedence = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")

obd_data_joined = obd_data.join(
    sku_supercedence.select("SKU", "SKUSTATUS"), 
    obd_data["CURRENT_OBD_SKU"] == sku_supercedence["SKU"], 
    how='left'
)

obd_data_active_skus = obd_data_joined.filter(F.lower(F.col("SKUSTATUS")) == 'active') \
                                        .select("CURRENT_OBD_SKU", "PREVIOUS_OBD_SKU") 


actual_ecr_sales = actual_ecr_sales.join(obd_data_active_skus, actual_ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"], how="left")
actual_ecr_sales = actual_ecr_sales.with_column("SKU", F.coalesce(F.col("CURRENT_OBD_SKU"), F.col("SKU")))

In [21]:
actual_ecr_sales.select(F.sum("NET_SALES").alias("TOTAL_SALES")).show()

-----------------
|"TOTAL_SALES"  |
-----------------
|298086.000000  |
-----------------

